# 02 Volatility & VaR Modeling
Fitting GARCH models and estimating Value-at-Risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.garch_models import fit_garch
from src.var_estimation import compute_rolling_var, calculate_risk_labels

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
log_returns = pd.read_csv('../data/processed/log_returns.csv', index_col=0, parse_dates=True)
primary = '^STOXX50E'
returns = log_returns[primary] * 100 # Rescale for stability if needed, but model handles it

print("Fitting GJR-GARCH(1,1) with Student-t distribution...")
res = fit_garch(returns, model='GJR-GARCH')
print(res.summary())

In [ ]:
vol = res.conditional_volatility / 100 # Scale back
returns_scaled = returns / 100

nu = res.params['nu']
var_95 = compute_rolling_var(returns_scaled, vol, alpha=0.05, nu=nu)
var_99 = compute_rolling_var(returns_scaled, vol, alpha=0.01, nu=nu)

plt.figure(figsize=(12, 6))
plt.plot(returns_scaled, label='Actual Returns', color='gray', alpha=0.5)
plt.plot(-var_95, label='VaR 95%', color='orange')
plt.plot(-var_99, label='VaR 99%', color='red')
plt.title('Returns vs VaR Estimates')
plt.legend()
plt.savefig('../results/plots/var_vs_returns.png')
plt.show()

In [ ]:
labels, threshold = calculate_risk_labels(returns_scaled, var_95)
print(f"Risk Threshold (c): {threshold:.4f}")
print(f"High Risk States: {labels.sum()} / {len(labels)} ({labels.mean()*100:.2f}%)")

processed_df = pd.DataFrame({
    'returns': returns_scaled,
    'volatility': vol,
    'var_95': var_95,
    'risk_label': labels
}, index=returns_scaled.index)

processed_df.to_csv('../data/processed/risk_data.csv')
print("Saved risk data for RL training.")